In [ ]:
import os
import re
import json
import hashlib
from datetime import datetime

import jieba
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer


OUTPUT_ROOT = "./outputs"
DATA_PATH = "./data/text_data.xlsx"
STOP_FILE = "./data/stopwords.txt"

TOP_N_TOPICS_FOR_YEARLY_PLOT = 10
INCLUDE_OUTLIER_TOPIC = False

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = os.path.join(OUTPUT_ROOT, f"bertopic_bgem3_{TIMESTAMP}")
CACHE_DIR = os.path.join(OUTPUT_ROOT, "_cache_embeddings")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_PATH}")
print(f"Cache directory: {CACHE_DIR}")


def load_stopwords(file_path: str) -> set:
    stop_list = []
    try:
        with open(file_path, encoding="utf-8") as f:
            stop_list = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Stopword file not found: {file_path}")
    return set(stop_list)


stopwords = load_stopwords(STOP_FILE)

custom_words = ["一带一路", "若开邦", "昂山素季", "吴温纳貌伦", "孟中印缅"]
for w in custom_words:
    jieba.add_word(w)


def chinese_word_cut(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    words = jieba.lcut(text)
    filtered = []

    for word in words:
        if not word or not word.strip():
            continue
        if word in stopwords:
            continue
        if len(word) == 1:
            continue
        if any(ch.isdigit() for ch in word):
            continue
        if re.match(r"^[a-zA-Z]+$", word) and len(word) < 3:
            continue
        filtered.append(word)

    return " ".join(filtered)


def bgem3_semantic_segmentation(raw_text: str):
    if not raw_text or not raw_text.strip():
        return []

    raw_text = raw_text.strip()
    text_length = len(raw_text)
    estimated_tokens = text_length * 2

    if estimated_tokens <= 3000:
        return [raw_text]

    sentence_delimiters = r'([。！？；;.!?]|\n\n|\r\n\r\n)'
    parts = re.split(sentence_delimiters, raw_text)

    sentences = []
    current_sentence = ""

    for part in parts:
        if not part or not part.strip():
            continue
        if re.match(sentence_delimiters, part):
            if current_sentence:
                current_sentence += part
                sentences.append(current_sentence.strip())
                current_sentence = ""
        else:
            current_sentence += part

    if current_sentence.strip():
        sentences.append(current_sentence.strip())

    segments = []
    current_segment = []
    current_char_count = 0
    max_chars_per_segment = 1200

    for sent in sentences:
        sent_chars = len(sent)

        if sent_chars > 800:
            if current_segment:
                segments.append(" ".join(current_segment))
                current_segment = []
                current_char_count = 0
            segments.append(sent)
            continue

        if current_char_count + sent_chars > max_chars_per_segment:
            if current_segment:
                segments.append(" ".join(current_segment))
            current_segment = [sent]
            current_char_count = sent_chars
        else:
            current_segment.append(sent)
            current_char_count += sent_chars

    if current_segment:
        segments.append(" ".join(current_segment))

    final_segments = []
    for seg in segments:
        if len(seg) > 1500:
            chunk_size = 1200
            chunks = [seg[i:i + chunk_size] for i in range(0, len(seg), chunk_size)]
            final_segments.extend([c.strip() for c in chunks if c.strip()])
        else:
            final_segments.append(seg.strip())

    print(f"Text length: {text_length} chars -> {len(final_segments)} segments")
    return final_segments


def extract_year_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    m = re.search(r"(19\d{2}|20\d{2})", s)
    if m:
        return int(m.group(1))
    return np.nan


print("Loading data...")
data = pd.read_excel(DATA_PATH)
print(f"Loaded {len(data)} documents")

text_col = data.columns[0]

if "Year" not in data.columns:
    raise ValueError("The input Excel file must contain a column named 'Year'.")

data["Year_clean"] = data["Year"].apply(extract_year_value)

if data["Year_clean"].isna().mean() > 0.2:
    print("Warning: a relatively high proportion of Year values could not be parsed.")

print(
    f"Year range: {int(np.nanmin(data['Year_clean']))} - "
    f"{int(np.nanmax(data['Year_clean']))}"
)

print("Segmenting raw texts and preprocessing segments...")

raw_segments = []
processed_segments = []
original_indices = []
segment_types = []
document_titles = []
segment_years = []

for idx, row in data.iterrows():
    raw_text = str(row[text_col]) if pd.notna(row[text_col]) else ""
    raw_text = raw_text.strip()
    if not raw_text:
        continue

    year_int = row["Year_clean"]

    if len(data.columns) > 1:
        doc_title = str(row[data.columns[1]]) if pd.notna(row[data.columns[1]]) else f"Document_{idx}"
    else:
        doc_title = f"Document_{idx}"

    segs_raw = bgem3_semantic_segmentation(raw_text)
    if not segs_raw:
        continue

    segs_proc = [chinese_word_cut(s) for s in segs_raw]
    keep_pairs = [(r, p) for r, p in zip(segs_raw, segs_proc) if p.strip()]

    if not keep_pairs:
        continue

    segs_raw, segs_proc = zip(*keep_pairs)
    segs_raw, segs_proc = list(segs_raw), list(segs_proc)

    raw_segments.extend(segs_raw)
    processed_segments.extend(segs_proc)
    original_indices.extend([idx] * len(segs_raw))
    document_titles.extend([doc_title] * len(segs_raw))
    segment_years.extend([year_int] * len(segs_raw))

    for j in range(len(segs_raw)):
        if len(segs_raw) == 1:
            segment_types.append("whole")
        elif j == 0:
            segment_types.append("beginning")
        elif j == len(segs_raw) - 1:
            segment_types.append("ending")
        else:
            segment_types.append("middle")

    if (idx + 1) % 20 == 0:
        print(f"Processed {idx + 1}/{len(data)} documents")

print(
    f"Documents: {len(data)} | Segments: {len(raw_segments)} | "
    f"Avg segments per document: {len(raw_segments) / len(data):.1f}"
)

segment_years = pd.Series(segment_years, dtype="float")

print("Loading BGE-M3...")
embedding_model = SentenceTransformer(
    "BAAI/bge-m3",
    trust_remote_code=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

embedding_model.max_seq_length = 4096

print("BGE-M3 loaded")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"Max sequence length: {embedding_model.max_seq_length}")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


def get_embeddings_bgem3(texts, batch_size=12):
    embeddings_list = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        batch_num = start // batch_size + 1

        try:
            emb = embedding_model.encode(
                batch,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True
            )
            embeddings_list.append(emb)
            if batch_num % 5 == 0 or batch_num == total_batches:
                print(f"Embedding batches: {batch_num}/{total_batches}")
        except Exception as e:
            print(f"Batch {batch_num} failed, falling back to single-item encoding: {e}")
            for t in batch:
                try:
                    one = embedding_model.encode(
                        [t],
                        show_progress_bar=False,
                        normalize_embeddings=True,
                        convert_to_numpy=True
                    )
                    embeddings_list.append(one)
                except Exception:
                    dim = embedding_model.get_sentence_embedding_dimension()
                    embeddings_list.append(np.zeros((1, dim), dtype=np.float32))

    return np.vstack(embeddings_list) if embeddings_list else np.array([])


def segments_fingerprint(texts):
    h = hashlib.md5()
    for t in texts:
        h.update(t.encode("utf-8", errors="ignore"))
        h.update(b"\n")
    return h.hexdigest()


print("Computing or loading embeddings...")
fp = segments_fingerprint(raw_segments)
emb_cache_path = os.path.join(CACHE_DIR, f"embeddings_bgem3_{fp}.npy")
emb_meta_path = os.path.join(CACHE_DIR, f"embeddings_bgem3_{fp}.json")

if os.path.exists(emb_cache_path):
    print(f"Loading cached embeddings: {emb_cache_path}")
    embeddings = np.load(emb_cache_path)
    print(f"Embeddings shape: {embeddings.shape}")
else:
    print("No cached embeddings found. Computing embeddings...")
    embeddings = get_embeddings_bgem3(raw_segments, batch_size=12)
    np.save(emb_cache_path, embeddings)

    meta = {
        "model": "BAAI/bge-m3",
        "max_seq_length": int(embedding_model.max_seq_length),
        "num_segments": int(len(raw_segments)),
        "batch_size": 12,
        "fingerprint": fp,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    with open(emb_meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"Saved embeddings cache: {emb_cache_path}")
    print(f"Saved embedding metadata: {emb_meta_path}")


print("Configuring BERTopic...")

umap_model = UMAP(
    n_neighbors=15,
    n_components=8,
    min_dist=0.0,
    metric="cosine"
)

hdbscan_model = HDBSCAN(
    min_cluster_size=11,
    min_samples=6,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 3),
    min_df=3,
    max_df=1.0,
    stop_words=list(stopwords),
    max_features=5000
)

ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True,
    reduce_frequent_words=True
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics="auto",
    calculate_probabilities=True,
    verbose=True,
    top_n_words=10
)

print("Training BERTopic...")
topics, probabilities = topic_model.fit_transform(processed_segments, embeddings)

unique_topics = set(topics)
num_topics = len([t for t in unique_topics if t != -1])
num_noise = sum(1 for t in topics if t == -1)

print(f"Topic modeling completed: {num_topics} topics, {num_noise} outlier segments ({num_noise / len(processed_segments):.1%})")


print("Aligning probability columns with topic IDs...")

if probabilities is None:
    prob_topic_ids = []
    print("Warning: probabilities is None. Document-level distributions will fall back to hard assignment.")
else:
    topic_info = topic_model.get_topic_info()
    topic_id_list = topic_info["Topic"].tolist()
    topic_ids_no_outlier = [t for t in topic_id_list if t != -1]

    if probabilities.shape[1] == len(topic_ids_no_outlier):
        prob_topic_ids = topic_ids_no_outlier
    elif probabilities.shape[1] == len(topic_id_list):
        prob_topic_ids = topic_id_list
    else:
        fallback = sorted([t for t in topic_model.get_topics().keys() if t != -1])
        if len(fallback) != probabilities.shape[1]:
            raise ValueError(
                f"Cannot align probability columns ({probabilities.shape[1]}) "
                f"with topic IDs."
            )
        prob_topic_ids = fallback

    print(f"Probability columns: {probabilities.shape[1]} | Mapped topic IDs: {len(prob_topic_ids)}")


doc_info = pd.DataFrame({
    "Document_Index": original_indices,
    "Document_Title": document_titles,
    "Year": segment_years,
    "Segment_Type": segment_types,
    "Primary_Topic": topics,
    "Segment_Raw": raw_segments,
    "Segment_Processed": processed_segments
})

doc_info.to_excel(os.path.join(OUTPUT_PATH, "segment_analysis.xlsx"), index=False)
print("Saved: segment_analysis.xlsx")


print("Generating document_topic_matrix...")

all_topic_ids = sorted(list(topic_model.get_topics().keys()))
if not INCLUDE_OUTLIER_TOPIC:
    all_topic_ids = [t for t in all_topic_ids if t != -1]

doc_indices_unique = sorted(doc_info["Document_Index"].unique().tolist())
segment_doc_idx = np.array(doc_info["Document_Index"].tolist())

doc_topic_sum = {d: {tid: 0.0 for tid in all_topic_ids} for d in doc_indices_unique}

if probabilities is None or len(prob_topic_ids) == 0:
    for i, t in enumerate(topics):
        doc_idx = segment_doc_idx[i]
        if (not INCLUDE_OUTLIER_TOPIC) and (t == -1):
            continue
        if t in doc_topic_sum[doc_idx]:
            doc_topic_sum[doc_idx][t] += 1.0
else:
    probs_arr = np.array(probabilities)
    for i in range(probs_arr.shape[0]):
        doc_idx = segment_doc_idx[i]
        row = probs_arr[i]
        for col, p in enumerate(row):
            tid = prob_topic_ids[col]
            if (not INCLUDE_OUTLIER_TOPIC) and (tid == -1):
                continue
            if tid in doc_topic_sum[doc_idx]:
                doc_topic_sum[doc_idx][tid] += float(p)

doc_rows = []
for doc_idx in doc_indices_unique:
    segs = doc_info[doc_info["Document_Index"] == doc_idx]
    year_val = segs["Year"].iloc[0]
    title_val = segs["Document_Title"].iloc[0]

    row = {
        "Document_Index": doc_idx,
        "Year": year_val,
        "Document_Title": title_val
    }

    total = sum(doc_topic_sum[doc_idx].values())

    if total > 0:
        for tid in all_topic_ids:
            row[f"Topic_{tid}"] = doc_topic_sum[doc_idx][tid] / total
    else:
        for tid in all_topic_ids:
            row[f"Topic_{tid}"] = 0.0

    doc_rows.append(row)

doc_topic_df = pd.DataFrame(doc_rows)
doc_topic_df.to_excel(os.path.join(OUTPUT_PATH, "document_topic_matrix.xlsx"), index=False)
print("Saved: document_topic_matrix.xlsx")


print("Generating yearly topic composition plot...")

topic_cols = [c for c in doc_topic_df.columns if c.startswith("Topic_")]

yearly_base = doc_topic_df.dropna(subset=["Year"]).copy()
yearly_base["Year"] = yearly_base["Year"].astype(int)

yearly = (
    yearly_base
    .groupby("Year")[topic_cols]
    .mean()
    .sort_index()
)

if yearly.empty:
    print("Yearly aggregation is empty. Please check the Year column.")
else:
    N = TOP_N_TOPICS_FOR_YEARLY_PLOT
    topic_global_rank = yearly.mean(axis=0).sort_values(ascending=False)
    top_topics = topic_global_rank.head(min(N, len(topic_global_rank))).index.tolist()

    yearly_top = yearly[top_topics].copy()
    other_cols = [c for c in topic_cols if c not in top_topics]

    if other_cols:
        yearly_top["Other"] = yearly[other_cols].sum(axis=1)
    else:
        yearly_top["Other"] = 0.0

    ax = yearly_top.plot(kind="area", stacked=True, figsize=(12, 6))
    ax.set_title("Yearly Topic Composition (Document-Averaged, Soft Mixture)")
    ax.set_xlabel("Year")
    ax.set_ylabel("Share")
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_PATH, "Figure1_Yearly_Topic_Composition_StackedArea.png")
    plt.savefig(fig_path, dpi=300)
    plt.close()

    yearly_xlsx = os.path.join(OUTPUT_PATH, "yearly_topic_composition_document_avg.xlsx")
    yearly_top.to_excel(yearly_xlsx)

    print(f"Saved: {os.path.basename(fig_path)}")
    print(f"Saved: {os.path.basename(yearly_xlsx)}")


print("Generating topic_keywords.xlsx...")

topic_keywords = {}
for tid in sorted(topic_model.get_topics().keys()):
    words = topic_model.get_topic(tid)
    if words:
        topic_keywords[tid] = [w for w, _ in words[:10]]

df_keywords = pd.DataFrame.from_dict(topic_keywords, orient="index")
df_keywords.columns = [f"Keyword_{i + 1}" for i in range(df_keywords.shape[1])]
df_keywords.index.name = "Topic"
df_keywords.to_excel(os.path.join(OUTPUT_PATH, "topic_keywords.xlsx"))
print("Saved: topic_keywords.xlsx")


print("Generating BERTopic visualizations...")

TOP_N_TOPICS_FOR_VIZ = 50
top_n_topics_viz = min(TOP_N_TOPICS_FOR_VIZ, max(1, num_topics))


def safe_write_html(fig, filename):
    try:
        fig.write_html(os.path.join(OUTPUT_PATH, filename))
        print(f"Saved: {filename}")
    except Exception as e:
        print(f"Failed: {filename} | {e}")


safe_write_html(
    topic_model.visualize_barchart(top_n_topics=top_n_topics_viz),
    "viz_barchart.html"
)

safe_write_html(
    topic_model.visualize_topics(top_n_topics=top_n_topics_viz),
    "viz_topics_bubble.html"
)

safe_write_html(
    topic_model.visualize_hierarchy(top_n_topics=top_n_topics_viz),
    "viz_hierarchy.html"
)

try:
    if num_topics > 2:
        n_clusters = min(10, num_topics - 1)
        safe_write_html(
            topic_model.visualize_heatmap(n_clusters=n_clusters, top_n_topics=top_n_topics_viz),
            "viz_heatmap.html"
        )
    else:
        print("Skipping heatmap because the number of topics is too small.")
except Exception as e:
    print(f"Failed: viz_heatmap.html | {e}")

safe_write_html(
    topic_model.visualize_term_rank(),
    "viz_term_rank.html"
)

try:
    safe_write_html(
        topic_model.visualize_documents(
            processed_segments,
            topics=topics,
            embeddings=embeddings,
            hide_annotations=False
        ),
        "viz_documents.html"
    )
except Exception as e:
    print(f"Failed: viz_documents.html | {e}")


method_description = f"""
Output directory: {OUTPUT_PATH}

Analytical workflow:
- Segment-level topic modeling with BERTopic and calculate_probabilities=True
- Document-level topic mixtures obtained by aggregating segment-level probabilities
- Yearly topic composition obtained by averaging document-level topic mixtures by year

Reproducibility note:
- This pipeline contains stochastic components, especially in dimensionality reduction and clustering.
- Exact topic assignments may therefore vary slightly across runs.
- Such variation should not materially alter the interpretation of macro-level topic structure and temporal patterns.

Files generated:
- segment_analysis.xlsx
- document_topic_matrix.xlsx
- yearly_topic_composition_document_avg.xlsx
- topic_keywords.xlsx
- viz_barchart.html
- viz_topics_bubble.html
- viz_hierarchy.html
- viz_heatmap.html
- viz_term_rank.html
- viz_documents.html
- Figure1_Yearly_Topic_Composition_StackedArea.png
"""

with open(os.path.join(OUTPUT_PATH, "methodology_description.txt"), "w", encoding="utf-8") as f:
    f.write(method_description)

model_info = {
    "embedding_model": "BAAI/bge-m3",
    "max_seq_length": embedding_model.max_seq_length,
    "analysis_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "num_documents": int(len(data)),
    "num_segments": int(len(raw_segments)),
    "num_topics_excluding_outlier": int(num_topics),
    "include_outlier_in_doc_distribution": bool(INCLUDE_OUTLIER_TOPIC),
    "yearly_plot_top_n": int(TOP_N_TOPICS_FOR_YEARLY_PLOT),
    "analysis_method": "Per-segment soft distribution -> per-document distribution -> yearly composition",
    "reproducibility_note": "This pipeline includes stochastic components, so exact results may vary slightly across runs."
}

with open(os.path.join(OUTPUT_PATH, "model_config.json"), "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("Done")
print("=" * 60)
print(f"Results saved to: {OUTPUT_PATH}")